In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from torch.utils.data import random_split, DataLoader
from src.configs import SEED, BATCH_SIZE
from src.dataset import ImageDataset
from src.loss_function import Loss
from src.model import Model
from pathlib import Path
import torch

In [3]:
from torchvision.transforms import v2
import torch

# The mean and standard deviations across each channel for the normalized pixels
# of every single image in the "trainval" dataset
MEANS = (0.485, 0.456, 0.406)
STDS = (0.229, 0.224, 0.225)

trainval_transforms = v2.Compose([
    v2.Normalize(mean=MEANS, std=STDS)
])

In [4]:
annot_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annot_file_trainval, img_dir_trainval,
                                transform=trainval_transforms)

generator_ = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(trainval_dataset, [0.05, 0.95]
                                          ,generator=generator_)

len(train_dataset)

251

In [5]:
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                     num_workers=4, pin_memory=True, persistent_workers=True)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
model = Model().to(device, non_blocking=True)
loss_fn = Loss().to(device, non_blocking=True)

optimizer = torch.optim.Adam([
    {"params": model.backbone.parameters(), "lr": 5e-5},
    {"params": model.detector_head.parameters()}
    ], lr=1e-4)

In [ ]:
for epoch in range(20):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_dl:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        preds = model(X_batch)
        loss = loss_fn(preds, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(epoch, epoch_loss / len(train_dl))

0 95.85457468032837
1 45.30767583847046
2 28.812662363052368
3 20.82351291179657
4 16.03215003013611
5 13.665125966072083
6 11.672193884849548
7 10.832731485366821
8 9.91220510005951
9 9.208759009838104
10 8.741775035858154


In [ ]:
from src.inference_functions import compute_eval_stats

train_mAP = compute_eval_stats(model, train_dl, device)
train_mAP